# Make my Satellite1 Ultra parts

This page builds parts sized for **your** printer.

**You only need to do three things:**

1. Paste the code from the Calibrate step into the box below.
2. Click **Runtime → Run all** in the menu at the top.
3. Wait about 15 minutes, then your parts download automatically.

You will run this page **twice**: once with your round one code, which returns
the seven remaining test pieces at your scale, and once with your final code,
which returns the enclosure.

Nothing is installed on your computer. If a box asks you to allow something,
say yes — that is just Google asking permission to save the download.


In [ ]:
#@title Step 1 — paste your code here, then press the ▶ button { display-mode: "form" }
MY_CODE = ""  #@param {type:"string"}

import base64

# Calibration runs in two rounds and the code tells us which one you are in.
# S1U1- is round one: scale only, and what you get back is the seven remaining
# test pieces rebuilt at your scale. S1U- is the final code and returns the
# enclosure. The prefixes differ because the values alone cannot distinguish
# them: a printer whose every other measurement lands on nominal is a state a
# real machine can reach, and guessing wrong here means printing the wrong set.
ROUND_ONE_PREFIX = "S1U1-"
FINAL_PREFIX = "S1U-"

KEYS = ["xy_scale_correction_fraction","z_scale_correction_fraction","fastener_clearance_diameter_offset_mm","insert_bore_diameter_offset_mm","driver_cutout_diameter_offset_mm","passive_radiator_cutout_diameter_offset_mm","cable_passage_diameter_offset_mm","gasket_sheet_thickness_mm","gasket_compressed_thickness_offset_mm","active_driver_flange_thickness_mm","passive_radiator_flange_thickness_mm"]

if not MY_CODE.strip():
    raise SystemExit("Paste the code from the measuring step into the box above, then run this cell again.")

text = MY_CODE.strip()
# Longest prefix first: S1U- is a prefix of S1U1- only in the other direction,
# but checking the specific one first keeps this correct if either changes.
if text.startswith(ROUND_ONE_PREFIX):
    ROUND_ONE, payload = True, text[len(ROUND_ONE_PREFIX):]
elif text.startswith(FINAL_PREFIX):
    ROUND_ONE, payload = False, text[len(FINAL_PREFIX):]
else:
    raise SystemExit("That does not look like a code. It should start with S1U1- or S1U-")

try:
    padded = payload + "=" * (-len(payload) % 4)
    numbers = [float(v) for v in base64.b64decode(padded).decode().split(",")]
except Exception:
    raise SystemExit("That code looks damaged. Copy it again from the measuring step.")
if len(numbers) != len(KEYS):
    raise SystemExit("That code is incomplete. Copy it again from the measuring step.")

CALIBRATION = dict(zip(KEYS, numbers))
if ROUND_ONE:
    print("Round one code. You will get the seven remaining test pieces, at your scale.\n")
else:
    print("Final code. You will get the enclosure and the Satellite1 top parts.\n")
for key, value in CALIBRATION.items():
    print(f"  {key:45s} {value}")


## Step 2 — everything below runs on its own

You do not need to change anything here. Just let it finish.

In [ ]:
#@title Install the CAD engine (about 3 minutes)
!pip install --quiet cadquery==2.6.1 pyyaml==6.0.2 numpy==2.2.6 2>&1 | tail -2
print("CAD engine ready.")


In [ ]:
#@title Fetch the Satellite1 Ultra design
!rm -rf /content/Satellite1-Ultra
!git clone --depth 1 --quiet https://github.com/BigPappy098/Satellite1-Ultra.git /content/Satellite1-Ultra
%cd /content/Satellite1-Ultra
!pip install --quiet --no-deps -e . 2>&1 | tail -1
print("Design files ready.")


In [ ]:
#@title Check your numbers are safe
import yaml
from satellite1_ultra.configuration import validate_physical_calibration

validate_physical_calibration(CALIBRATION)   # refuses anything physically implausible

with open("config/physical_calibration.yaml", "w") as handle:
    handle.write("# Generated from your measurements.\n")
    for key, value in CALIBRATION.items():
        handle.write(f"{key}: {value}\n")
print("Your numbers passed every safety check.")


In [ ]:
#@title Build your parts (about 10 minutes — this is the slow one)
import time
start = time.time()
from pathlib import Path
from satellite1_ultra.configuration import load_design_parameters
from satellite1_ultra.exporting import export_parts

parameters = load_design_parameters()
written = export_parts(Path("exports"), parameters)
print(f"\nBuilt {len(written)} files in {(time.time()-start)/60:.1f} minutes.")


In [ ]:
#@title Package your parts and download them
import shutil, os
from pathlib import Path
from satellite1_ultra.builder_files import (
    CALIBRATION_STAGE_TWO, ULTRA_PRINT_ORDER, OFFICIAL_TOP_PRINT_ORDER)
from satellite1_ultra.official import OFFICIAL_PRINT_PARTS_REQUIRED

out = Path("/content/MY_SATELLITE1_ULTRA_PARTS")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True, exist_ok=True)

def copy_3mf(order, folder):
    (out / folder).mkdir(parents=True, exist_ok=True)
    for source, friendly, _quantity in order:
        shutil.copy2(Path("exports/3mf") / f"{source}.3mf", out / folder / friendly)
        shutil.copy2(Path("exports/stl") / f"{source}.stl",
                     out / folder / f"{Path(friendly).stem}.stl")

if ROUND_ONE:
    # Only the test pieces, and only the ones still to be printed. The part
    # that produced this code has already been printed and measured; handing it
    # back invites a builder to reprint it and re-measure a number they already
    # have.
    remaining = [row for row in CALIBRATION_STAGE_TWO
                 if row[0] != "coupon_official_interface"]
    copy_3mf(remaining, "PRINT_THESE_SEVEN")
    (out / "READ_ME.txt").write_text(
        "Round two test pieces, sized for your printer.\n\n"
        "PRINT_THESE_SEVEN - print all seven, one at a time, with the same\n"
        "                    settings you used for the first part. The last\n"
        "                    one is the flexible cable seal, so load TPU.\n\n"
        "Then go back to the website, open the Calibrate step, and fill in the\n"
        "'Measure those seven' tab. That gives you the final code.\n")
else:
    copy_3mf(ULTRA_PRINT_ORDER, "1_ENCLOSURE_PARTS")
    official = {part.name: part for part in OFFICIAL_PRINT_PARTS_REQUIRED}
    (out / "2_SATELLITE_TOP_PARTS").mkdir(parents=True, exist_ok=True)
    for source, friendly, _quantity in OFFICIAL_TOP_PRINT_ORDER:
        shutil.copy2(official[source].stl_path, out / "2_SATELLITE_TOP_PARTS" / friendly)
    (out / "READ_ME.txt").write_text(
        "Your parts, sized for your printer from the measurements you entered.\n\n"
        "1_ENCLOSURE_PARTS      - the enclosure. Print every file.\n"
        "2_SATELLITE_TOP_PARTS  - the original Satellite1 top. Print all six.\n\n"
        "Then go back to the website and follow the Build it step.\n")

archive = shutil.make_archive("/content/MY_SATELLITE1_ULTRA_PARTS", "zip", out.parent, out.name)
print(f"Ready: {os.path.getsize(archive)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
    print("\nYour download should start now.")
except Exception:
    print("\nOpen the folder icon on the left and download MY_SATELLITE1_ULTRA_PARTS.zip")


## Done

Your parts are in **MY_SATELLITE1_ULTRA_PARTS.zip**.

If the download did not start, click the folder icon on the left-hand side and
download it from there.

If this was your **round one** code, print those seven pieces and go back to the
website's Calibrate step to fill in the last tab. If it was your **final** code,
go to **Build it**.

*These files are generated from your measurements. Nothing has been physically
built and tested yet, so keep checking as you go.*
